# Data Quality — product

Jalankan **Run All** dengan kernel `env`. Keenam pemeriksaan di bawah memakai aturan yang sama untuk semua sumber. Hasil transaksi mengikuti kontrak canonical 12 kolom; `product_id` memakai SKU asli dari Product Master.

Product Master tetap berisi identitas produk dan harga referensi. Data sumber tetap utuh. Semua aturan dijalankan saat persiapan agar pemeriksaan duplikat sudah memperhitungkan hasil mapping produk; enam bagian berikut memperlihatkan hasil tiap pemeriksaan.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pipeline/validation/analysis.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.validation.analysis import (
    load_analysis, standard_data, missing_values, quality_issues,
    type_report, summary, export_analysis,
)

SOURCE = "product"
hasil = load_analysis(SOURCE, ROOT / "data/source")
data_bersih = standard_data(hasil)


## 1. Missing value

Field wajib: order ID, tanggal, produk, quantity, harga satuan, status; total transaksi Website juga wajib. Semua field master wajib. Baris yang tidak memenuhi syarat ditolak. Customer/kota/email yang tidak tersedia tetap kosong, tanpa dummy. Field opsional kosong yang tersedia dalam source dicatat sebagai warning.

In [2]:
display(missing_values(hasil))
display(quality_issues(hasil, "missing"))

,sumber,kolom,jumlah_kosong
0,product,sku,0
1,product,product_name,0
2,product,brand,0
3,product,category,0
4,product,price,0


,sumber,baris,tingkat,kolom,masalah,nilai_asli


## 2. Duplicate

Business key: `(channel, order_id)` untuk dataset satu item per order saat ini; master memakai `product_id`/SKU. Record yang sama disimpan satu kali. Jika key sama tetapi nilainya berbeda, semua versi ditolak untuk ditinjau. Bila kelak order memiliki beberapa item, tambahkan line ID dari sumber.

In [3]:
display(quality_issues(hasil, "duplicate"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli


## 3. Invalid value

Quantity wajib integer positif. Harga wajib positif dan maksimal dua desimal. Total harus sama dengan quantity × harga satuan. Status dataset saat ini: `Completed`, `Cancelled`, `Returned`; status lain ditolak. Harga berbeda dari master diberi warning karena mungkin promo. Total harga adalah nilai bruto; hitung penjualan selesai hanya dari `Completed`.

In [4]:
display(quality_issues(hasil, "invalid"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli


## 4. Date format

Tanggal Shopee/Tokopedia: DD/MM/YYYY; Website: MMM DD, YYYY; Offline: DD-MMM-YYYY. Hasil CSV selalu YYYY-MM-DD. Tanggal tidak valid ditolak. Product Master tidak memiliki tanggal transaksi.

In [5]:
display(quality_issues(hasil, "date"))
if "tanggal_order" in data_bersih:
    display(data_bersih[["order_id", "tanggal_order"]].head(5))
else:
    print("Tidak berlaku: master produk tidak memiliki tanggal transaksi.")

,sumber,baris,tingkat,kolom,masalah,nilai_asli


Tidak berlaku: master produk tidak memiliki tanggal transaksi.


## 5. Data type

Identifier string, quantity Int64, tanggal datetime, dan uang Decimal. CSV tidak menyimpan tipe data; ekspor tanggal menggunakan YYYY-MM-DD. Teks dirapikan spasinya tanpa merusak nama brand, SKU, shade, atau SPF/PA++++.

In [6]:
display(type_report(data_bersih))

,kolom,dtype,tipe_nilai
0,product_id,string,str
1,product_name,string,str
2,brand,string,str
3,kategori,string,str
4,harga_satuan,object,Decimal


## 6. Product consistency

Variasi huruf besar/kecil, spasi, underscore, dan hyphen dicocokkan ke master. Nama produk dan kategori mengikuti master. Produk tidak dikenal/typo ambigu ditolak, tanpa menebak SKU. Pada master, SKU harus unik.

In [7]:
display(data_bersih[["product_id", "product_name", "kategori"]].drop_duplicates())
display(quality_issues(hasil, "product"))

,product_id,product_name,kategori
0,WRD-SKC-001,Wardah UV Shield Aqua Fresh Essence SPF 50 PA+...,Skincare
1,WRD-SKC-002,Wardah Lightening Micellar Water 100ml,Skincare
2,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup
3,WRD-MUP-002,Wardah Colorfit Perfect Glow Cushion 13N,Makeup
4,EMN-SKC-001,Emina Bright Stuff Face Wash 100ml,Skincare
5,EMN-SKC-002,Emina Sun Battle SPF 50 PA++++ 30ml,Skincare
6,EMN-MUP-001,Emina Cheek Lit Cream Blush Peach,Makeup
7,EMN-MUP-002,Emina Glossy Stain 01 Autumn Bell,Makeup
8,MKO-MUP-001,Make Over Powerstay Weightless Liquid Foundati...,Makeup
9,MKO-MUP-002,Make Over Powerstay Matte Powder Foundation N20,Makeup


,sumber,baris,tingkat,kolom,masalah,nilai_asli


## Hasil akhir

Format dan urutan kolom sama untuk semua transaksi. `kota` adalah kota pelanggan online atau kota toko offline; Website yang tidak memiliki kota tetap kosong. Nama channel tetap Shopee/Tokopedia/Website/Offline Store agar bisa dibandingkan.

Hasil utama: `data/processed/clean/`. Notebook ini menyimpan file bersih sumber yang dibahas; `analisa.ipynb` menyimpan seluruh sumber, `sales.csv`, `summary.csv`, serta satu `quality_issues.csv` untuk detail masalah.

In [8]:
ringkasan = summary(hasil)
assert (ringkasan["awal"] == ringkasan["bersih"] + ringkasan["duplikat"] + ringkasan["ditolak"]).all()
display(ringkasan)
display(data_bersih.head(10))
folder_hasil = export_analysis(hasil)
print("Tersimpan:", folder_hasil)

,sumber,awal,bersih,duplikat,ditolak
0,product,20,20,0,0


,product_id,product_name,brand,kategori,harga_satuan
0,WRD-SKC-001,Wardah UV Shield Aqua Fresh Essence SPF 50 PA+...,Wardah,Skincare,68900.00
1,WRD-SKC-002,Wardah Lightening Micellar Water 100ml,Wardah,Skincare,28900.00
2,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Wardah,Makeup,62900.00
3,WRD-MUP-002,Wardah Colorfit Perfect Glow Cushion 13N,Wardah,Makeup,109000.00
4,EMN-SKC-001,Emina Bright Stuff Face Wash 100ml,Emina,Skincare,27900.00
5,EMN-SKC-002,Emina Sun Battle SPF 50 PA++++ 30ml,Emina,Skincare,49900.00
6,EMN-MUP-001,Emina Cheek Lit Cream Blush Peach,Emina,Makeup,46900.00
7,EMN-MUP-002,Emina Glossy Stain 01 Autumn Bell,Emina,Makeup,52900.00
8,MKO-MUP-001,Make Over Powerstay Weightless Liquid Foundati...,Make Over,Makeup,169000.00
9,MKO-MUP-002,Make Over Powerstay Matte Powder Foundation N20,Make Over,Makeup,179000.00


Tersimpan: C:\Users\ADVAN\OneDrive - Universitas Teknologi Yogyakarta\Rinaldi\Ecommerce Sales\data\processed\clean
